# Day 1: Large Language Models & Prompt Engineering

**Workshop: Introduction to Generative AI & LLMs**

---

## Learning Objectives

By the end of this session, you will be able to:

1. Understand the generative AI landscape and key milestones
2. Explain the Transformer architecture at a high level
3. Apply prompt engineering techniques: zero-shot, few-shot, chain-of-thought, role prompting
4. Use the OpenAI API for chat completions, streaming, and function calling
5. Control generation with temperature and top-p parameters
6. Build reusable prompt templates for common NLP tasks

---

## 1. The Generative AI Landscape

### A Brief Timeline

| Year | Milestone | Significance |
|------|-----------|-------------|
| 2017 | "Attention Is All You Need" (Vaswani et al.) | Introduced the Transformer architecture |
| 2018 | GPT-1 (OpenAI) | Demonstrated generative pre-training for NLP |
| 2018 | BERT (Google) | Bidirectional pre-training; dominated NLU benchmarks |
| 2019 | GPT-2 (OpenAI) | 1.5B parameters; raised concerns about misuse |
| 2020 | GPT-3 (OpenAI) | 175B parameters; few-shot learning capabilities |
| 2021 | DALL-E, Codex (OpenAI) | Image generation and code generation from text |
| 2022 | ChatGPT (OpenAI), Stable Diffusion | Conversational AI goes mainstream; open-source image generation |
| 2023 | GPT-4, LLaMA, Claude, PaLM 2 | Multimodal models; open-weight LLMs proliferate |
| 2024 | Mixtral, Gemini, Claude 3, LLaMA 3 | Mixture-of-experts; longer context windows; improved reasoning |
| 2025 | Claude 4, GPT-4.5, open-weight explosion | Agents, tool use, and real-world deployment at scale |

### Categories of Generative AI

- **Text Generation**: LLMs (GPT-4, Claude, LLaMA, Mistral)
- **Image Generation**: Diffusion models (Stable Diffusion, DALL-E 3, Midjourney)
- **Audio/Music**: Bark, MusicGen, Suno
- **Video**: Sora, Runway Gen-2
- **Code**: Codex, StarCoder, CodeLlama, Copilot
- **Multimodal**: GPT-4V, Gemini, Claude 3 (text + vision)

### Key Concepts

- **Pre-training**: Learning language patterns from massive corpora (unsupervised)
- **Fine-tuning**: Adapting a pre-trained model to a specific task or domain
- **RLHF**: Reinforcement Learning from Human Feedback, aligning models with human preferences
- **Inference**: Using a trained model to generate outputs
- **Tokens**: Sub-word units that LLMs process (roughly 4 characters per token in English)

---

## 2. Transformer Architecture Intuition

The Transformer is the foundation of all modern LLMs. Here is a conceptual overview:

### High-Level Architecture

```
Input Text
    |
    v
[Tokenizer] --> Token IDs --> [Embedding Layer] --> Vectors
    |                                                  |
    v                                                  v
[Positional Encoding] ----> Vectors with position info
    |
    v
+---------------------------------+
| Transformer Block (x N layers)  |
|                                 |
| [Multi-Head Self-Attention]     |
|      |                          |
| [Add & Layer Norm]              |
|      |                          |
| [Feed-Forward Network]          |
|      |                          |
| [Add & Layer Norm]              |
+---------------------------------+
    |
    v
[Output Head] --> Next token probabilities
```

### Self-Attention Mechanism

Self-attention lets each token "look at" every other token in the sequence to understand context.

For each token, the model computes:
- **Query (Q)**: "What am I looking for?"
- **Key (K)**: "What do I contain?"
- **Value (V)**: "What information do I provide?"

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

### Decoder-Only vs Encoder-Decoder

| Architecture | Examples | Use Case |
|-------------|----------|----------|
| Encoder-only | BERT, RoBERTa | Classification, NER, embeddings |
| Decoder-only | GPT, LLaMA, Claude | Text generation, chat |
| Encoder-Decoder | T5, BART | Translation, summarization |

Modern LLMs (GPT-4, Claude, LLaMA) are **decoder-only** Transformers that predict the next token autoregressively.

---

## 3. Setup

Let's install the required packages and configure our API key.

In [ ]:
# Install required packages
!pip install openai tiktoken python-dotenv --quiet

In [ ]:
import os
import json
from textwrap import dedent

from openai import OpenAI
import tiktoken

# Load API key from environment variable
# Set it before running: export OPENAI_API_KEY="sk-..."
# Or create a .env file with OPENAI_API_KEY=sk-...
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# Verify connection
print("OpenAI client initialized.")
print(f"API key configured: {'Yes' if os.environ.get('OPENAI_API_KEY') else 'No -- please set OPENAI_API_KEY'}")

In [ ]:
# Helper function for chat completions (we will reuse this throughout)
def chat(messages, model="gpt-4o-mini", temperature=0.7, max_tokens=1024, **kwargs):
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        **kwargs,
    )
    return response.choices[0].message.content


def quick_chat(user_prompt, system_prompt="You are a helpful assistant.", **kwargs):
    """Convenience wrapper: single user message with optional system prompt."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    return chat(messages, **kwargs)

### Token Counting

Understanding tokens is important for cost estimation and context window management.

In [ ]:
# Token counting with tiktoken
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

sample_texts = [
    "Hello, world!",
    "The Transformer architecture was introduced in 2017.",
    "Reinforcement Learning from Human Feedback (RLHF) is a technique used to align language models.",
    "Bonjour le monde! Les modeles de langage sont fascinants.",
]

for text in sample_texts:
    tokens = encoding.encode(text)
    print(f"Text: {text!r}")
    print(f"  Tokens ({len(tokens)}): {tokens}")
    print(f"  Decoded: {[encoding.decode_single_token_bytes(t) for t in tokens]}")
    print()

---

## 4. Prompt Engineering Techniques

Prompt engineering is the practice of designing effective inputs to get desired outputs from LLMs. It is the most accessible and cost-effective way to control LLM behavior.

### 4.1 Zero-Shot Prompting

Zero-shot prompting provides no examples. The model relies entirely on its pre-trained knowledge.

In [ ]:
# Zero-shot: Classification
zero_shot_prompt = """Classify the following text into one of these categories: 
[Technology, Science, Politics, Sports, Entertainment]

Text: "The new quantum processor achieved a breakthrough in error correction, 
reducing noise by 90% compared to previous generations."

Category:"""

result = quick_chat(zero_shot_prompt, temperature=0)
print("Zero-shot classification result:")
print(result)

In [ ]:
# Zero-shot: Sentiment analysis
zero_shot_sentiment = """Analyze the sentiment of the following review. 
Respond with exactly one word: Positive, Negative, or Neutral.

Review: "The hotel was decent but the breakfast was disappointing. 
The staff made up for it with excellent service."

Sentiment:"""

result = quick_chat(zero_shot_sentiment, temperature=0)
print(f"Sentiment: {result}")

### 4.2 Few-Shot Prompting

Few-shot prompting provides examples to guide the model's output format and reasoning.

In [ ]:
# Few-shot: Named Entity Recognition
few_shot_ner = """Extract named entities from the text. Format: ENTITY (TYPE)

Example 1:
Text: "Apple CEO Tim Cook announced the new iPhone at the Cupertino headquarters."
Entities: Apple (ORG), Tim Cook (PERSON), iPhone (PRODUCT), Cupertino (LOCATION)

Example 2:
Text: "President Macron met with Chancellor Scholz in Berlin on Tuesday."
Entities: Macron (PERSON), Scholz (PERSON), Berlin (LOCATION), Tuesday (DATE)

Example 3:
Text: "Tesla's stock rose 5% after the Q3 earnings report exceeded Wall Street expectations."
Entities: Tesla (ORG), 5% (PERCENTAGE), Q3 (DATE), Wall Street (LOCATION)

Now extract entities from:
Text: "Dr. Sarah Chen from MIT published a paper on CRISPR gene editing in Nature last March."
Entities:"""

result = quick_chat(few_shot_ner, temperature=0)
print("Few-shot NER result:")
print(result)

In [ ]:
# Few-shot: Translation with style
few_shot_translation = """Translate the following English sentences to French, 
using a formal academic register.

English: "This study examines the impact of climate change on biodiversity."
French: "La presente etude examine l'impact du changement climatique sur la biodiversite."

English: "Our results suggest a significant correlation between the two variables."
French: "Nos resultats suggerent une correlation significative entre les deux variables."

English: "The methodology was validated using cross-validation techniques."
French:"""

result = quick_chat(few_shot_translation, temperature=0.3)
print("Few-shot translation:")
print(result)

### 4.3 Chain-of-Thought (CoT) Prompting

Chain-of-thought prompting encourages the model to break down complex reasoning step-by-step, significantly improving accuracy on math, logic, and multi-step tasks.

In [ ]:
# Without Chain-of-Thought
no_cot_prompt = """A store sells notebooks for $3 each. If you buy 5 or more, 
you get a 20% discount on the total. Tax is 8%. 
How much do 7 notebooks cost after tax?

Answer:"""

result_no_cot = quick_chat(no_cot_prompt, temperature=0)
print("WITHOUT Chain-of-Thought:")
print(result_no_cot)
print()

In [ ]:
# With Chain-of-Thought
cot_prompt = """A store sells notebooks for $3 each. If you buy 5 or more, 
you get a 20% discount on the total. Tax is 8%. 
How much do 7 notebooks cost after tax?

Let's think step by step:"""

result_cot = quick_chat(cot_prompt, temperature=0)
print("WITH Chain-of-Thought:")
print(result_cot)

In [ ]:
# CoT for logical reasoning
cot_logic = """Solve this logic puzzle step by step.

Three friends -- Alice, Bob, and Carol -- each have a different pet: a cat, a dog, 
and a fish. We know:
1. Alice does not have a cat.
2. Bob does not have a dog.
3. The person with the fish is not Carol.

Let's reason through this step by step to determine who has which pet:"""

result = quick_chat(cot_logic, temperature=0)
print("CoT Logic Puzzle:")
print(result)

### 4.4 Role Prompting and System Prompts

System prompts set the persona, constraints, and behavior of the model. They are powerful for creating specialized assistants.

In [ ]:
# Role: Academic Paper Reviewer
reviewer_system = """You are an experienced academic peer reviewer in computer science. 
When reviewing, you:
- Assess the clarity of the abstract
- Evaluate the claimed contributions
- Identify potential weaknesses or gaps
- Provide constructive suggestions
- Rate the abstract on a scale of 1-5 for clarity, novelty, and significance

Be thorough but constructive. Use academic language."""

abstract = """Review this abstract:

"We present a novel approach to sentiment analysis using graph neural networks (GNNs). 
Unlike traditional methods that treat text as a sequence, we construct a word co-occurrence 
graph and apply message-passing to capture long-range dependencies. Our method achieves 
state-of-the-art results on three benchmark datasets, outperforming BERT-based models 
by 2-3% in accuracy while using 60% fewer parameters."""

messages = [
    {"role": "system", "content": reviewer_system},
    {"role": "user", "content": abstract},
]

review = chat(messages, temperature=0.5)
print("Peer Review:")
print(review)

In [ ]:
# Role: Socratic Tutor
tutor_system = """You are a Socratic tutor for machine learning concepts. 
You NEVER give direct answers. Instead, you:
1. Ask guiding questions to help the student discover the answer
2. Provide hints when they are stuck
3. Correct misconceptions gently
4. Celebrate when they reach the right understanding

Keep responses concise (2-3 sentences max per turn)."""

# Simulate a multi-turn conversation
conversation = [
    {"role": "system", "content": tutor_system},
    {"role": "user", "content": "What is overfitting?"},
]

# Turn 1
reply1 = chat(conversation, temperature=0.7)
print(f"Tutor: {reply1}")
conversation.append({"role": "assistant", "content": reply1})

# Turn 2: Student responds
student_reply = "Is it when the model memorizes the training data?"
print(f"\nStudent: {student_reply}")
conversation.append({"role": "user", "content": student_reply})

reply2 = chat(conversation, temperature=0.7)
print(f"\nTutor: {reply2}")

### 4.5 Structured Output (JSON Mode)

Getting LLMs to produce structured output is critical for integrating them into pipelines.

In [ ]:
# JSON mode: Extract structured data from text
json_system = """You are a data extraction assistant. 
Always respond with valid JSON. No other text."""

json_prompt = """Extract the following information from this job posting as JSON:

Job Posting: "Senior Machine Learning Engineer at TechCorp (San Francisco, CA). 
Remote-friendly. Salary: $180,000 - $250,000. Requirements: 5+ years of experience 
in ML/AI, proficiency in Python and PyTorch, experience with distributed training. 
Nice to have: publications in top-tier venues, experience with LLMs."

Extract: title, company, location, remote, salary_min, salary_max, 
required_skills (list), nice_to_have (list), min_years_experience"""

result = quick_chat(
    json_prompt,
    system_prompt=json_system,
    temperature=0,
    response_format={"type": "json_object"},
)

# Parse and pretty-print
parsed = json.loads(result)
print(json.dumps(parsed, indent=2))

In [ ]:
# JSON mode: Multi-item extraction
multi_extract_prompt = """Extract all research papers mentioned in this text as a JSON 
object with a key "papers" containing a list of objects.

Each paper object should have: title, authors (list), year, venue.

Text: "Recent advances in NLP include the Transformer model introduced by Vaswani et al. 
in their 2017 paper 'Attention Is All You Need' published at NeurIPS. This was followed 
by Devlin et al.'s 'BERT: Pre-training of Deep Bidirectional Transformers' at NAACL 2019. 
More recently, Touvron et al. released 'LLaMA: Open and Efficient Foundation Language Models' 
as an arXiv preprint in 2023."""

result = quick_chat(
    multi_extract_prompt,
    system_prompt=json_system,
    temperature=0,
    response_format={"type": "json_object"},
)

papers = json.loads(result)
print(json.dumps(papers, indent=2))

---

## 5. OpenAI API Deep Dive

### 5.1 Chat Completions

In [ ]:
# Full chat completion with all response metadata
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise assistant. Reply in 1-2 sentences."},
        {"role": "user", "content": "Explain what an embedding is in machine learning."},
    ],
    temperature=0.5,
    max_tokens=150,
)

# Inspect the full response object
print("Model:", response.model)
print("Content:", response.choices[0].message.content)
print("Finish reason:", response.choices[0].finish_reason)
print("\nToken usage:")
print(f"  Prompt tokens:     {response.usage.prompt_tokens}")
print(f"  Completion tokens: {response.usage.completion_tokens}")
print(f"  Total tokens:      {response.usage.total_tokens}")

### 5.2 Streaming Responses

Streaming lets you display tokens as they arrive, improving perceived latency for the user.

In [ ]:
# Streaming chat completion
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about machine learning."},
    ],
    temperature=0.9,
    max_tokens=50,
    stream=True,
)

print("Streaming response:")
collected_content = ""
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content is not None:
        print(delta.content, end="", flush=True)
        collected_content += delta.content

print("\n\nFull collected response:", collected_content)

### 5.3 Function Calling (Tool Use)

Function calling allows the model to request calls to external functions you define. This is the foundation for building AI agents.

In [ ]:
# Define tools the model can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'Paris'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_papers",
            "description": "Search for academic papers by topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search query for papers",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
]

# Simulated function implementations
def get_weather(city, unit="celsius"):
    """Simulated weather API."""
    # In production, this would call a real weather API
    mock_data = {
        "paris": {"temp": 18, "condition": "Partly cloudy"},
        "tokyo": {"temp": 24, "condition": "Sunny"},
        "new york": {"temp": 15, "condition": "Rainy"},
    }
    data = mock_data.get(city.lower(), {"temp": 20, "condition": "Unknown"})
    if unit == "fahrenheit":
        data["temp"] = data["temp"] * 9 / 5 + 32
    return json.dumps({"city": city, "temperature": data["temp"], "unit": unit, "condition": data["condition"]})


def search_papers(query, max_results=5):
    """Simulated paper search."""
    return json.dumps({
        "query": query,
        "results": [
            {"title": f"Paper about {query} #{i+1}", "year": 2024, "citations": (5 - i) * 100}
            for i in range(min(max_results, 3))
        ],
    })


# Map function names to implementations
available_functions = {
    "get_weather": get_weather,
    "search_papers": search_papers,
}

print("Tools and mock functions defined.")

In [ ]:
# Function calling: let the model decide which tool to call
messages = [
    {"role": "system", "content": "You are a helpful research assistant with access to weather and paper search tools."},
    {"role": "user", "content": "What's the weather like in Paris? Also, find me papers about transformer architectures."},
]

# Step 1: Send message with tools
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

assistant_message = response.choices[0].message
print("Model wants to call these tools:")

# Step 2: Execute each tool call
if assistant_message.tool_calls:
    messages.append(assistant_message)  # Add assistant's response to conversation

    for tool_call in assistant_message.tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        print(f"  -> {func_name}({func_args})")

        # Call the function
        func_result = available_functions[func_name](**func_args)

        # Add tool result to conversation
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": func_result,
        })

    # Step 3: Get final response incorporating tool results
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )

    print("\nFinal response:")
    print(final_response.choices[0].message.content)
else:
    print("No tool calls made.")
    print(assistant_message.content)

---

## 6. Temperature and Top-p Exploration

These parameters control the randomness/creativity of generation:

- **Temperature** (0.0 - 2.0): Lower = more deterministic; higher = more creative/random
- **Top-p** (0.0 - 1.0): Nucleus sampling. Lower = considers fewer tokens; higher = considers more

In [ ]:
# Compare outputs at different temperatures
prompt = "Write a one-sentence description of artificial intelligence."
temperatures = [0.0, 0.5, 1.0, 1.5]

print("=" * 80)
print(f"Prompt: {prompt}")
print("=" * 80)

for temp in temperatures:
    print(f"\n--- Temperature: {temp} ---")
    # Generate 3 samples at each temperature to show variance
    for i in range(3):
        result = quick_chat(prompt, temperature=temp, max_tokens=60)
        print(f"  Sample {i+1}: {result.strip()}")

In [ ]:
# Compare top-p values
prompt = "Complete this story in one sentence: The robot opened its eyes for the first time and"

top_p_values = [0.1, 0.5, 0.9, 1.0]

print(f"Prompt: {prompt}")
print("=" * 80)

for top_p in top_p_values:
    print(f"\n--- Top-p: {top_p} ---")
    for i in range(3):
        result = quick_chat(prompt, temperature=1.0, max_tokens=60, top_p=top_p)
        print(f"  Sample {i+1}: {result.strip()}")

### Practical Guidelines for Temperature and Top-p

| Task | Temperature | Top-p | Rationale |
|------|-------------|-------|-----------|
| Code generation | 0.0 - 0.2 | 0.1 - 0.3 | Deterministic, correct code |
| Data extraction | 0.0 | 1.0 | Factual accuracy |
| Summarization | 0.3 - 0.5 | 0.9 | Faithful but fluent |
| Creative writing | 0.8 - 1.2 | 0.9 - 1.0 | Diverse, creative output |
| Brainstorming | 1.0 - 1.5 | 1.0 | Maximum diversity |
| Classification | 0.0 | 1.0 | Consistent, reproducible |

---

## 7. Lab: Reusable Prompt Templates

In this lab, we build a library of reusable prompt templates for common NLP tasks: summarization, information extraction, and classification.

In [ ]:
class PromptTemplate:
    """A reusable prompt template with variable substitution."""

    def __init__(self, template: str, system_prompt: str = "You are a helpful assistant."):
        self.template = template
        self.system_prompt = system_prompt

    def format(self, **kwargs) -> str:
        """Fill in the template variables."""
        return self.template.format(**kwargs)

    def run(self, model="gpt-4o-mini", temperature=0, max_tokens=1024, **kwargs):
        """Format the template and send to the API."""
        user_message = self.format(**kwargs)
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message},
        ]
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return response.choices[0].message.content

    def run_json(self, model="gpt-4o-mini", temperature=0, max_tokens=1024, **kwargs):
        """Format, send, and parse JSON response."""
        user_message = self.format(**kwargs)
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_message},
        ]
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            response_format={"type": "json_object"},
        )
        return json.loads(response.choices[0].message.content)


print("PromptTemplate class defined.")

In [ ]:
# Template 1: Summarization
summarizer = PromptTemplate(
    template=dedent("""\
        Summarize the following text in {style} style.
        Target length: {length}.

        Text:
        {text}

        Summary:"""),
    system_prompt="You are an expert summarizer. Produce clear, accurate summaries.",
)

sample_text = """
Large Language Models (LLMs) have demonstrated remarkable capabilities in natural language 
understanding and generation. These models, trained on vast corpora of text data, can perform 
a wide range of tasks including translation, summarization, question answering, and creative 
writing. Recent advances have focused on improving efficiency through techniques like 
quantization and pruning, making it possible to deploy these models on consumer hardware. 
However, challenges remain around hallucination, bias, and the environmental cost of training. 
Researchers are actively working on alignment techniques such as RLHF and constitutional AI 
to make these models safer and more reliable.
"""

# Academic summary
result = summarizer.run(text=sample_text, style="academic", length="2-3 sentences")
print("Academic Summary:")
print(result)

print("\n" + "="*60 + "\n")

# Bullet-point summary
result = summarizer.run(text=sample_text, style="bullet-point", length="3-5 bullet points")
print("Bullet-Point Summary:")
print(result)

In [ ]:
# Template 2: Information Extraction
extractor = PromptTemplate(
    template=dedent("""\
        Extract the following fields from the text below.
        Return the result as a JSON object with these keys: {fields}
        If a field is not found, use null.

        Text:
        {text}"""),
    system_prompt="You are a precise data extraction assistant. Always respond with valid JSON only.",
)

# Extract from a research paper description
paper_text = """
In their 2023 paper "Scaling Data-Constrained Language Models", Muennighoff et al. from 
Hugging Face investigate how to best train LLMs when data is limited. They find that 
repeating data up to 4 epochs causes negligible loss in performance, but beyond that, 
returns diminish. The study uses models ranging from 400M to 9B parameters and was 
accepted at NeurIPS 2023.
"""

result = extractor.run_json(
    text=paper_text,
    fields="title, authors, year, venue, model_sizes, key_finding",
)
print("Extracted paper metadata:")
print(json.dumps(result, indent=2))

In [ ]:
# Template 3: Text Classification
classifier = PromptTemplate(
    template=dedent("""\
        Classify the following text into exactly one of these categories: {categories}

        Return a JSON object with:
        - "category": the chosen category
        - "confidence": your confidence (high/medium/low)
        - "reasoning": one sentence explaining your choice

        Text:
        {text}"""),
    system_prompt="You are a text classification system. Always respond with valid JSON only.",
)

# Classify multiple texts
texts_to_classify = [
    "The new CRISPR technique allows for precise gene editing without off-target effects.",
    "The central bank raised interest rates by 25 basis points amid inflation concerns.",
    "The latest superhero movie broke box office records with a $200M opening weekend.",
    "A new study shows that regular exercise reduces the risk of Alzheimer's by 40%.",
]

categories = "Science, Technology, Finance, Politics, Health, Entertainment, Sports"

print("Classification Results:")
print("=" * 60)
for text in texts_to_classify:
    result = classifier.run_json(text=text, categories=categories)
    print(f"\nText: {text[:70]}...")
    print(f"  Category:   {result.get('category')}")
    print(f"  Confidence: {result.get('confidence')}")
    print(f"  Reasoning:  {result.get('reasoning')}")

In [ ]:
# Template 4: Batch Processing Pipeline
# Combine templates into a pipeline that processes a document end-to-end

def analyze_document(text: str) -> dict:
    """Run a full analysis pipeline on a document."""
    # Step 1: Summarize
    summary = summarizer.run(text=text, style="concise", length="2 sentences")

    # Step 2: Extract metadata
    metadata = extractor.run_json(
        text=text,
        fields="topic, key_entities, date_references, sentiment",
    )

    # Step 3: Classify
    classification = classifier.run_json(
        text=text,
        categories="Research, News, Opinion, Tutorial, Review",
    )

    return {
        "summary": summary,
        "metadata": metadata,
        "classification": classification,
    }


# Run the pipeline
doc = """
In a landmark study published in Nature in October 2024, researchers from DeepMind 
demonstrated that their AlphaFold 3 model can predict not only protein structures but 
also protein-ligand interactions with near-experimental accuracy. This breakthrough 
could accelerate drug discovery by reducing the need for costly wet-lab experiments. 
The model was trained on a curated dataset of 100,000 protein-ligand complexes and 
uses a diffusion-based architecture. Critics note that the model's predictions still 
need experimental validation and that access to the model remains limited.
"""

analysis = analyze_document(doc)

print("Full Document Analysis")
print("=" * 60)
print(f"\nSummary: {analysis['summary']}")
print(f"\nMetadata: {json.dumps(analysis['metadata'], indent=2)}")
print(f"\nClassification: {json.dumps(analysis['classification'], indent=2)}")

---

## Summary

In this session, we covered:

1. **GenAI Landscape**: Timeline from Transformers (2017) to modern multimodal models
2. **Transformer Architecture**: Self-attention, Q/K/V mechanism, decoder-only design
3. **Prompt Engineering**: Zero-shot, few-shot, chain-of-thought, role prompting, JSON mode
4. **OpenAI API**: Chat completions, streaming, function calling
5. **Sampling Parameters**: Temperature and top-p for controlling output diversity
6. **Prompt Templates**: Reusable patterns for summarization, extraction, and classification

### Next Session

In Day 2, we will build on these foundations to create **Retrieval-Augmented Generation (RAG)** systems using embeddings, vector databases, and LangChain.